# Load data

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/movie_final.csv")

In [ ]:
df.shape

(6591, 11)

In [ ]:
# Đếm số lượng missing mỗi cột
df.isnull().sum()

,0
id,0
title,0
budget,2230
revenue,2217
release_date,28
genres,137
vote_average,0
vote_count,0
runtime,13
original_language,0


In [ ]:
df = df.dropna(subset=['revenue']).reset_index(drop=True)

In [ ]:
df = df.drop(columns=['title', 'id', 'original_language','vote_count','vote_average'])

In [ ]:
import numpy as np

df.loc[df['budget'] == 0, 'budget'] = np.nan
df.loc[df['runtime'] == 0, 'runtime'] = np.nan

In [ ]:
df = df.dropna(subset=['release_date']).reset_index(drop=True)

In [ ]:
import numpy as np

df['director'] = df['director'].replace('Other', np.nan)

In [ ]:
df.columns

Index(['budget', 'revenue', 'release_date', 'genres', 'runtime', 'director'], dtype='object')

In [ ]:
# Đếm số lượng missing mỗi cột
df.isnull().sum()

,0
budget,383
revenue,0
release_date,0
genres,130
runtime,46
director,89


In [ ]:
df.shape

(4368, 13)

# Kiểm định **Budget**

In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'])
df['year'] = df['release_date'].dt.year
df['month'] = df['release_date'].dt.month
df['day'] = df['release_date'].dt.day

In [ ]:
df['budget_missing'] = df['budget'].isnull().astype(int)

In [ ]:
from scipy.stats import ttest_ind

num_cols = ['revenue', 'runtime', 'year', 'month', 'day']

for col in num_cols:
    group1 = df[df['budget_missing'] == 0][col]
    group2 = df[df['budget_missing'] == 1][col]

    stat, p = ttest_ind(group1, group2, nan_policy='omit')
    print(f"{col}: p-value = {p}")

revenue: p-value = 0.0008670989294284209
runtime: p-value = 0.008185063250854392
year: p-value = 0.0014281040274318425
month: p-value = 0.39454706214456503
day: p-value = 0.7001626738254686


In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['director'], df['budget_missing'])

chi2, p, dof, expected = chi2_contingency(contingency)

print("p-value:", p)

p-value: 2.712599994258141e-06


=> `budget` không là MCAR vì `revenue`, `runtime`, `year`, `director`  
Vì `director` có nhiều categories hiếm -> loại `director` ra khi check MAR

In [ ]:
df['budget_missing'].value_counts(normalize=True)

,proportion
budget_missing,
0,0.912317
1,0.087683


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =========================
# COPY DATAFRAME
# =========================
df_mar = df.copy()

# =========================
# TARGET
# =========================
df_mar['budget_missing'] = df_mar['budget'].isna().astype(int)

# =========================
# NUMERIC CLEAN
# =========================
num_cols = ['revenue', 'runtime', 'year']

for col in num_cols:
    df_mar[col] = pd.to_numeric(df_mar[col], errors='coerce')

# =========================
# REVENUE LOG TRANSFORM
# =========================
df_mar.loc[df_mar['revenue'] < 0, 'revenue'] = np.nan

df_mar['revenue'] = np.log1p(df_mar['revenue'])

# =========================
# FILL MISSING
# =========================
for col in num_cols:
    df_mar[col] = df_mar[col].fillna(df_mar[col].median())

# =========================
# BUILD X
# =========================
X = df_mar[['revenue', 'runtime', 'year']].astype(float)

# add constant
X = sm.add_constant(X)

# target
y = df_mar['budget_missing']

# =========================
# LOGIT MODEL
# =========================
model = sm.Logit(y, X)

result = model.fit()

# =========================
# SUMMARY
# =========================
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.292378
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:         budget_missing   No. Observations:                 4368
Model:                          Logit   Df Residuals:                     4364
Method:                           MLE   Df Model:                            3
Date:                Thu, 07 May 2026   Pseudo R-squ.:                 0.01604
Time:                        16:16:35   Log-Likelihood:                -1277.1
converged:                       True   LL-Null:                       -1297.9
Covariance Type:            nonrobust   LLR p-value:                 4.789e-09
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        223.1441     65.004      3.433      0.001      95.739     350.549
revenue       -0.0726      0.

# Kiểm định **Runtime**

In [ ]:
df['runtime_missing'] = df['runtime'].isnull().astype(int)

In [ ]:
from scipy.stats import ttest_ind

num_cols = ['budget', 'revenue', 'year', 'month', 'day']

for col in num_cols:

    group1 = df[df['runtime_missing'] == 0][col].dropna()
    group2 = df[df['runtime_missing'] == 1][col].dropna()

    print(f"\n{col}")

    stat, p = ttest_ind(group1, group2)

    print(f"p-value = {p}")


budget
p-value = 0.7466044577947699

revenue
p-value = 0.17212424520960126

year
p-value = 0.8196442630036647

month
p-value = 0.9836532106203382

day
p-value = 0.1573091163432356


In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['director'], df['runtime_missing'])

chi2, p, dof, expected = chi2_contingency(contingency)

print("p-value:", p)

p-value: 0.031983014279289


=> `runtime` không là MCAR vì `director`

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =========================
# COPY DATAFRAME
# =========================
df_mar = df.copy()

# =========================
# TARGET
# =========================
df_mar['runtime_missing'] = df_mar['runtime'].isna().astype(int)

# =========================
# DIRECTOR FREQUENCY ENCODING
# =========================
df_mar['director'] = df_mar['director'].fillna('Unknown')

freq = df_mar['director'].value_counts()

df_mar['director_freq'] = df_mar['director'].map(freq)

# =========================
# BUILD X
# =========================
X = df_mar[['director_freq']].astype(float)

X = sm.add_constant(X)

# target
y = df_mar['runtime_missing']

# =========================
# MODEL
# =========================
model = sm.Logit(y, X)

result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.055897
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:        runtime_missing   No. Observations:                 4368
Model:                          Logit   Df Residuals:                     4366
Method:                           MLE   Df Model:                            1
Date:                Thu, 07 May 2026   Pseudo R-squ.:                 0.04332
Time:                        16:16:35   Log-Likelihood:                -244.16
converged:                       True   LL-Null:                       -255.21
Covariance Type:            nonrobust   LLR p-value:                 2.575e-06
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -4.7597      0.166    -28.646      0.000      -5.085      -4.434
director_freq     0.

=> `runtime` là MAR theo `director`

# Kiểm định **Director**

In [ ]:
df['director_missing'] = df['director'].isnull().astype(int)

In [ ]:
num_cols = ['budget', 'revenue', 'runtime', 'year', 'month', 'day']

for col in num_cols:
    group1 = df[df['director_missing'] == 0][col]
    group2 = df[df['director_missing'] == 1][col]

    stat, p = ttest_ind(group1, group2, nan_policy='omit')
    print(f"{col}: p-value = {p}")

budget: p-value = 0.6510587278289935
revenue: p-value = 0.06669748901563939
runtime: p-value = 6.038271102634536e-05
year: p-value = 0.5824226614843426
month: p-value = 0.5669686578536048
day: p-value = 0.9282648326991946


=> `director` không là MCAR vì `runtime`  
=> `director` là MNAR vì có những phim không có thông tin đạo diễn chính thức

# Kiểm định **Genres**

In [ ]:
df['genres_missing'] = df['genres'].isnull().astype(int)

In [ ]:
num_cols = ['budget', 'revenue', 'runtime', 'year', 'month', 'day']

for col in num_cols:
    group1 = df[df['genres_missing'] == 0][col]
    group2 = df[df['genres_missing'] == 1][col]

    stat, p = ttest_ind(group1, group2, nan_policy='omit')
    print(f"{col}: p-value = {p}")

budget: p-value = 0.5668440919797056
revenue: p-value = 0.020313498766656328
runtime: p-value = 3.0912460938979995e-34
year: p-value = 0.2366001987537589
month: p-value = 0.8797339437422761
day: p-value = 0.4087583538117364


In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['director'], df['genres_missing'])

chi2, p, dof, expected = chi2_contingency(contingency)

print("p-value:", p)

p-value: 0.00019314157135661062


=> `genres` không là MCAR vì `revenue`, `runtime`, `director`  
=> `genres` là MNAR vì có những phim không có thông tin thể loại chính thức